# Stage 1 (Text‑Only) Phishing Detector — Code Walkthrough + CLI Commands


It explains:
- what each file does (`loader.py`, `build_features.py`, `train.py`, `predict.py`, `cli.py`)
- what happens when you train vs predict
- what the printed metrics mean (high level)
- which **CLI commands** you can run now (Windows/macOS)

> Stage 1 = **text-only baseline** (TF‑IDF + Logistic Regression).


## 1) The Stage 1 pipeline in one sentence
**CSV (`text`, `label`) → load → TF‑IDF vectorise → Logistic Regression train → evaluate → save artifacts → predict later from artifacts.**

Why this baseline?
- Fast to train (seconds)
- Easy to explain in a report
- Gives an interpretable “starting point” before heavier models (metadata, hybrid, BERT)


## 2) What each file does
### `src/phishingdet/data/loader.py`
**Goal:** load the dataset from `data/raw/...` and return a clean dataframe with exactly:
- `text` (email content as string)
- `label` (0 = legit, 1 = phishing)

Key point: this is **cross-platform** because it finds the repo root using `Path(__file__)...` instead of hardcoding `F:\...`.

---

### `src/phishingdet/features/build_features.py`
**Goal:** convert text → numbers using **TF‑IDF**.

- `fit_vectorizer(train_texts)` learns the vocabulary on training data only and transforms the training texts.
- `transform_vectorizer(vectorizer, test_texts)` uses the same vocabulary to transform new texts.

Important: TF‑IDF uses your settings:
- `max_features=5000` (caps vocabulary size)
- `ngram_range=(1,2)` (single words + two-word phrases)

---

### `src/phishingdet/models/train.py`
**Goal:** train + evaluate + save outputs you can screenshot for the report.

Steps:
1) Load dataset (texts + labels)
2) Split into train/test (`test_size=0.2` once dataset is large enough)
3) Fit TF‑IDF on train, transform test
4) Train Logistic Regression (`model.fit(...)`)
5) Evaluate metrics: accuracy/precision/recall/F1 + confusion matrix + ROC‑AUC + PR‑AUC
6) Save:
   - `artifacts/model.joblib`
   - `artifacts/vectorizer.joblib`
   - `artifacts/results.json` + `artifacts/results_<timestamp>.json`
   - `artifacts/top_features.csv`

Also includes:
- **Top features** (most influential tokens)
- **Label shuffle sanity check** (should be around 0.5)

---

### `src/phishingdet/models/predict.py`
**Goal:** load saved artifacts and make predictions on new text **without retraining**.

It:
1) loads `model.joblib` + `vectorizer.joblib`
2) transforms text with TF‑IDF
3) returns:
   - prediction (0/1)
   - phishing probability (if available)
   - decision band (`legit` / `phishing` / `uncertain`)

---

### `src/phishingdet/cli.py`
**Goal:** make the project runnable via simple commands like:
- `phishingdet train ...`
- `phishingdet predict --text "..."`


## 3) Key Points
- Stage 1 is a **text-only baseline** using TF‑IDF + Logistic Regression.
- Training produces **saved artifacts** (model + vectorizer) so prediction is reproducible and doesn’t retrain.
- Evaluation outputs are saved as timestamped JSON (`results_<timestamp>.json`) for comparison across experiments.
- The dataset I used is currently synthetic/templated, so results can look unrealistically perfect (all 1.0).
- I added a **label-shuffle sanity check** — accuracy drops near 0.5, which suggests no leakage in the pipeline.
- Next, I’ll validate on a more realistic dataset, then move to Stage 2 (metadata) and Stage 3 (hybrid).


## 4) Train vs Predict (important distinction)
### Training (`train.py`)
- Reads CSV
- Creates a new vectorizer
- Trains a new model
- Saves new artifacts (overwrites old ones unless you timestamp them)

So yes: **every run retrains from scratch** and saves a fresh model.

### Prediction (`predict.py`)
- Does **not** read the CSV
- Loads model/vectorizer artifacts
- Predicts on any input text


## 5) Why metrics can be 1.0 on the 50k synthetic dataset
This is often because the dataset is **too separable**:
- phishing templates contain tokens like `http`, `verify`, `account`, `urgent`
- legit templates contain tokens like `thanks`, `meeting`, `cheers`

TF‑IDF + Logistic Regression can learn a near-perfect boundary on that style of data.

The label-shuffle check being ~0.5 is a good sign: it suggests the pipeline isn’t accidentally leaking labels into features.


## 6) CLI commands
### Activate the virtual environment
**Windows (PowerShell)**
```powershell
cd F:\GitHub\fyp-phishing-email-detection
.\.venv\Scripts\Activate.ps1
```

**macOS**
```bash
cd ~/GitHub/fyp-phishing-email-detection
source .venv/bin/activate
```

### Ensure the console command exists
From repo root (inside venv):
```bash
pip install -e .
```

### Run training (via CLI)
Console script (if installed):
```bash
phishingdet train --test_size 0.2 --random_state 42 --max_features 5000
```

Module form (always works):
```bash
python -m phishingdet.cli train --test_size 0.2 --random_state 42 --max_features 5000
```

### Run a single prediction
Console script:
```bash
phishingdet predict --text "Urgent: verify your account at http://example.com"
```

Module form:
```bash
python -m phishingdet.cli predict --text "Urgent: verify your account at http://example.com"
```

### Legacy commands
```bash
python -m phishingdet.models.train
python -m phishingdet.models.predict
```


## 7) What artifacts are created (for screenshots + report evidence)
After training, you should see:

- `artifacts/model.joblib` (trained Logistic Regression)
- `artifacts/vectorizer.joblib` (TF‑IDF vocab + settings)
- `artifacts/results.json` (latest run summary)
- `artifacts/results_<timestamp>.json` (historical runs)
- `artifacts/top_features.csv` (top weighted tokens)
